# Silver Layer Notebook

Parse Bronze data, apply validation rules, and write cleaned Delta data to Silver layer.

In [9]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, coalesce, to_date, when, round as spark_round, current_timestamp
from pyspark.sql.types import DoubleType, LongType

In [10]:
MINIO_ENDPOINT = "http://localhost:9010"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"
MINIO_BUCKET = "crypto-warehouse"

import sys
import subprocess

# Keep Spark and Delta versions compatible
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pyspark==4.0.0", "delta-spark==4.0.0"])

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Stop any existing session so new dependency/config set is applied
try:
    spark.stop()
except Exception:
    pass

builder = (
    SparkSession.builder.appName("DataWarehouse-ETL")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    # Delta Lake (explicitly required)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # MinIO (S3A)
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)

extra_packages = [
    "org.apache.hadoop:hadoop-aws:3.4.1",
    "software.amazon.awssdk:bundle:2.31.58",
]

spark = configure_spark_with_delta_pip(builder, extra_packages=extra_packages).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("spark.sql.extensions =", spark.conf.get("spark.sql.extensions", "<missing>"))
print("spark.sql.catalog.spark_catalog =", spark.conf.get("spark.sql.catalog.spark_catalog", "<missing>"))
spark

26/04/23 12:47:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/23 12:47:35 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


spark.sql.extensions = io.delta.sql.DeltaSparkSessionExtension
spark.sql.catalog.spark_catalog = org.apache.spark.sql.delta.catalog.DeltaCatalog


In [11]:
BASE_URI = f"s3a://{MINIO_BUCKET}"
BRONZE_PATH = f"{BASE_URI}/bronze"
SILVER_PATH = f"{BASE_URI}/silver"
REJECTED_PATH = f"{BASE_URI}/silver/ohlcv_rejected"

print("MINIO_ENDPOINT:", MINIO_ENDPOINT)
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)

MINIO_ENDPOINT: http://localhost:9010
BRONZE_PATH: s3a://crypto-warehouse/bronze
SILVER_PATH: s3a://crypto-warehouse/silver


In [20]:
bronze = spark.read.format("delta").load(BRONZE_PATH)
print("Bronze rows:", bronze.count())
bronze.select("Date").show(5, truncate=False)

Bronze rows: 7058
+----------+
|Date      |
+----------+
|01-01-2018|
|02-01-2018|
|03-01-2018|
|04-01-2018|
|05-01-2018|
+----------+
only showing top 5 rows


In [24]:
parsed = (
    bronze
    .withColumnRenamed("Market Cap", "market_cap")
    .withColumn("trade_date", to_date(col("Date"), "dd-MM-yyyy"))
    .withColumn("trade_date", when(col("trade_date").isNull(), to_date(col("Date"), "yyyy-MM-dd")).otherwise(col("trade_date")))
    .withColumn("open_price", col("Open").cast(DoubleType()))
    .withColumn("high_price", col("High").cast(DoubleType()))
    .withColumn("low_price", col("Low").cast(DoubleType()))
    .withColumn("close_price", col("Close").cast(DoubleType()))
    # .withColumn("adj_close_price", coalesce(col("Adj Close"), col("Close")).cast(DoubleType()))
    .withColumn("volume", col("Volume").cast(LongType()))
    .withColumn("market_cap", col("market_cap").cast(DoubleType()))
)
print("Parsed rows:", parsed.count())

Parsed rows: 7058


In [ ]:
from pyspark.sql.functions import coalesce, lit

valid_condition = coalesce((
    col("trade_date").isNotNull()
    & col("open_price").isNotNull()
    & col("high_price").isNotNull()
    & col("low_price").isNotNull()
    & col("close_price").isNotNull()
    & col("volume").isNotNull()
    & col("market_cap").isNotNull()
    & (col("open_price") > 0)
    & (col("high_price") > 0)
    & (col("low_price") > 0)
    & (col("close_price") > 0)
), lit(False))

silver = (
    parsed
    .filter(valid_condition)
    .withColumn("daily_return", spark_round((col("close_price") - col("open_price")) / col("open_price"), 8))
    .withColumn("price_range", spark_round(col("high_price") - col("low_price"), 8))
    .withColumn("processed_ts", current_timestamp())
    .select(
        "asset_symbol", "trade_date",
        "open_price", "high_price", "low_price", "close_price",
        "volume", "market_cap",
        "daily_return", "price_range",
        "source_file", "ingestion_ts", "processed_ts"
    )
)

rejected = parsed.filter(~valid_condition)
print("Parse rows:", parsed.count())
print("Silver rows:", silver.count())
print("Rejected rows:", rejected.count())

Parse rows: 7058
Silver rows: 7045
Rejected rows: 13


In [34]:
# SILVER_PATH.parent.mkdir(parents=True, exist_ok=True)
# REJECTED_PATH.parent.mkdir(parents=True, exist_ok=True)
silver.write.format("delta").mode("overwrite").save(SILVER_PATH)
rejected.write.format("delta").mode("overwrite").save(REJECTED_PATH)

print("Silver rows:", silver.count())
print("Rejected rows:", rejected.count())

Silver rows: 7045
Rejected rows: 13


In [35]:
silver.orderBy(col("trade_date").desc()).show(20, truncate=False)

+------------+----------+-----------+-----------+-----------+-----------+------+---------------+------------+-----------+--------------+--------------------------------+--------------------------+
|asset_symbol|trade_date|open_price |high_price |low_price  |close_price|volume|market_cap     |daily_return|price_range|source_file   |ingestion_ts                    |processed_ts              |
+------------+----------+-----------+-----------+-----------+-----------+------+---------------+------------+-----------+--------------+--------------------------------+--------------------------+
|LTC         |2024-07-31|71.69      |72.84      |70.11      |70.19      |NULL  |NULL           |-0.02092342 |2.73       |LITECOIN24.csv|2026-04-23T05:08:56.526699+00:00|2026-04-23 13:04:01.914842|
|LTC         |2024-07-30|73.75      |74.48      |71.39      |71.69      |NULL  |NULL           |-0.0279322  |3.09       |LITECOIN24.csv|2026-04-23T05:08:56.526699+00:00|2026-04-23 13:04:01.914842|
|LTC         |2

In [36]:
from pathlib import Path
import shutil


def _resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "BatchProcessor").exists():
            return candidate
    return cwd


repo_root = _resolve_repo_root()
export_dir = repo_root / "dataset" / "silver"
export_dir.mkdir(parents=True, exist_ok=True)

temp_dir = export_dir / "_silver_tmp"
output_csv = export_dir / "silver.csv"

if temp_dir.exists():
    shutil.rmtree(temp_dir)
if output_csv.exists():
    output_csv.unlink()

(
    silver.coalesce(1)
    .write.mode("overwrite")
    .option("header", "true")
    .csv(str(temp_dir))
)

part_files = list(temp_dir.glob("part-*.csv"))
if not part_files:
    raise FileNotFoundError("No CSV part file generated for silver dataset")

shutil.move(str(part_files[0]), str(output_csv))
shutil.rmtree(temp_dir)

print(f"Silver dataset exported to: {output_csv}")

Silver dataset exported to: /home/bnguyen/Desktop/finnhub-streaming-pipeline/dataset/silver/silver.csv
